In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# unit and sanity checks!
"""--------------------------------------------"""
# add pooled r2
# check if r2 changes as a function of trial count
# - yeah, definitely

# change epoch of movement avg
# add licks

# replicate neurotheory plots
# > check the fits for different regularization constants
# define responsive
# one regressor
# add time
"""--------------------------------------------"""

## init

In [ ]:
from sg.models import Encoder, StrategyEncoder

ns = [20, 50, 100, 200, 203]

encoders = {}

for n in ns:
    encoder = Encoder(subj_id, sess_id, n=n)
    encoder.get_r2()
    encoders[n] = encoder

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

In [ ]:
import numpy as np

scores = {n: encoder.scores["encoder"] for n, encoder in encoders.items()}

np.mean(list(scores.values()), axis=1)

In [ ]:
scores = {
    model: {n: encoder.scores[model] for n, encoder in encoders.items()}
    for model in ["baseline", "encoder"]
}

colors = {"baseline": "#666666", "encoder": "#DD9C11"}

fig, ax = plt.subplots(tight_layout=True)
for model, s_ in scores.items():
    ax.errorbar(
        x=s_.keys(),
        y=np.mean(list(s_.values()), axis=1),
        yerr=np.std(list(s_.values()), axis=1),
        color=colors[model],
        capsize=2,
        label=model,
    )

ax.axhline(y=0, color="#000000", linestyle="-", linewidth=0.5)

ax.set_xlabel("n. trials")
ax.set_ylabel(r"$r^2$")
ax.legend()

In [ ]:
encoder.verify()

In [ ]:
# check that nans are still caught as different between mb/mf with out of pool averging

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_weights()

In [ ]:
# se = ShuffledEncoder(
#     subj_id,
#     sess_id,
#     tv_keys=[
#         "response",
#         "rewarded",
#         "block_side",
#         "strategy",
#         "response_prev",
#         "rewarded_prev",
#     ],
# )
# se.plot_cvr2()
# se.plot_dr2()|
# se.plot_bound_r2()

## experiment

In [ ]:
encoder.reg_idxs["DLS"].shape, encoder.reg_idxs["DMS"].shape

In [ ]:
# r2 between DMS and DLS
from core.viz import plot_kdes

# scores
scores = {
    f"{reg}, {model}": encoder.scores[model][encoder.reg_idxs[reg]]
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}
# scores_mb = {f"{reg}, {model}": encoder_mb.scores[model][encoder_mb.reg_idxs[reg]] for reg in encoder_mb.regions for model in ['baseline', 'encoder']}
scores_mf = {
    f"{reg}, {model}": encoder_mf.scores[model][encoder_mf.reg_idxs[reg]]
    for reg in encoder_mf.regions
    for model in ["baseline", "encoder"]
}

# styles
linestyles = {"baseline": "--", "encoder": "-"}
colors = {"DMS": "#562E9C", "DLS": "#009D51"}
styles = {
    f"{reg}, {model}": {"linestyle": linestyles[model], "color": colors[reg]}
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}

plot_kdes(scores, xlim=(-0.25, 1), line_kwargs=styles)
plot_kdes(scores_mf, xlim=(-0.25, 1), line_kwargs=styles)
# plot_kdes({f"{reg}, {model}": encoder_mb.scores[model][encoder_mb.reg_idxs[reg]] for reg in encoder_mb.regions for model in ['baseline', 'encoder']})